In [1]:
import numpy as np

# -----------------------------
# ОПИСАНИЕ СРЕДЫ
# -----------------------------

# Мир из 5 состояний: 0, 1, 2, 3, 4
n_states = 5

# Два действия:
# 0 = left
# 1 = right
n_actions = 2

# -----------------------------
# ПАРАМЕТРЫ ПОЛИТИКИ
# -----------------------------

# theta[state, action] — "сырые" параметры политики
# Это не вероятности, а числа, из которых вероятности будут получаться через softmax
theta = np.zeros((n_states, n_actions))

# -----------------------------
# ГИПЕРПАРАМЕТРЫ
# -----------------------------

# Скорость обучения
alpha = 0.1

# Насколько важны будущие награды
gamma = 0.9

# -----------------------------
# СРЕДА
# -----------------------------

def step(state, action):
    if action == 0:
        next_state = max(0, state - 1)
    else:
        next_state = min(n_states - 1, state + 1)

    reward = 1 if next_state == 4 else 0
    return next_state, reward

# -----------------------------
# SOFTMAX: ПРЕОБРАЗУЕМ theta В ВЕРОЯТНОСТИ
# -----------------------------

def softmax(x):
    # Вычитаем max для численной стабильности
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

# -----------------------------
# ПОЛУЧЕНИЕ ВЕРОЯТНОСТЕЙ ДЕЙСТВИЙ В СОСТОЯНИИ state
# -----------------------------

def policy_probs(state):
    return softmax(theta[state])

# -----------------------------
# ВЫБОР ДЕЙСТВИЯ ПО ПОЛИТИКЕ
# -----------------------------

def sample_action(state):
    probs = policy_probs(state)
    return np.random.choice(n_actions, p=probs)

# -----------------------------
# ОБУЧЕНИЕ
# -----------------------------

for episode in range(500):
    state = 0

    # Здесь будем хранить всю траекторию эпизода:
    # (state, action, reward)
    trajectory = []

    # Собираем один эпизод длиной до 20 шагов
    for _ in range(20):
        action = sample_action(state)
        next_state, reward = step(state, action)

        trajectory.append((state, action, reward))

        state = next_state

    # -----------------------------
    # СЧИТАЕМ RETURN G_t ДЛЯ КАЖДОГО ШАГА
    # -----------------------------

    returns = []
    G = 0

    # Идём с конца эпизода к началу:
    # G = r_{t+1} + gamma * G
    for _, _, reward in reversed(trajectory):
        G = reward + gamma * G
        returns.insert(0, G)

    # -----------------------------
    # REINFORCE UPDATE
    # -----------------------------

    for (state, action, _), G in zip(trajectory, returns):
        probs = policy_probs(state)

        # Градиент log policy для softmax-политики:
        #
        # для выбранного действия:
        # grad = 1 - pi(a|s)
        #
        # для остальных действий:
        # grad = -pi(a'|s)
        #
        # Удобно записать как:
        grad_log_pi = -probs
        grad_log_pi[action] += 1.0

        # Обновляем только строку theta[state]
        theta[state] += alpha * G * grad_log_pi

# Посмотрим на итоговые вероятности действий
for s in range(n_states):
    probs = policy_probs(s)
    print(f"state {s}: left={probs[0]:.3f}, right={probs[1]:.3f}")

state 0: left=0.035, right=0.965
state 1: left=0.002, right=0.998
state 2: left=0.009, right=0.991
state 3: left=0.012, right=0.988
state 4: left=0.000, right=1.000
